In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Dataset link and details: https://zenodo.org/records/5706578

In [ ]:
!wget https://zenodo.org/records/5706578/files/Train.zip
!wget https://zenodo.org/records/5706578/files/Val.zip

--2026-02-13 14:32:07--  https://zenodo.org/records/5706578/files/Train.zip
Resolving zenodo.org (zenodo.org)... 188.185.43.153, 188.184.103.118, 137.138.52.235, ...
Connecting to zenodo.org (zenodo.org)|188.185.43.153|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 4021669263 (3.7G) [application/octet-stream]
Saving to: ‘Train.zip’

Train.zip           100%[===================>]   3.75G  11.8MB/s    in 5m 34s  

2026-02-13 14:37:42 (11.5 MB/s) - ‘Train.zip’ saved [4021669263/4021669263]

--2026-02-13 14:37:42--  https://zenodo.org/records/5706578/files/Val.zip
Resolving zenodo.org (zenodo.org)... 188.184.103.118, 188.185.48.75, 188.185.43.153, ...
Connecting to zenodo.org (zenodo.org)|188.184.103.118|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 2425958254 (2.3G) [application/octet-stream]
Saving to: ‘Val.zip’

Val.zip             100%[===================>]   2.26G  11.5MB/s    in 5m 57s  

2026-02-13 14:43:40 (6.49 MB/s) - ‘Val.zi

In [ ]:
!unzip /content/Train.zip
!unzip /content/Val.zip

Streaming output truncated to the last 5000 lines.
 extracting: Train/Urban/images_png/2025.png  
 extracting: Train/Urban/images_png/2026.png  
 extracting: Train/Urban/images_png/2027.png  
 extracting: Train/Urban/images_png/2028.png  
 extracting: Train/Urban/images_png/2029.png  
 extracting: Train/Urban/images_png/2030.png  
  inflating: Train/Urban/images_png/2031.png  
 extracting: Train/Urban/images_png/2032.png  
 extracting: Train/Urban/images_png/2033.png  
  inflating: Train/Urban/images_png/2034.png  
  inflating: Train/Urban/images_png/2035.png  
 extracting: Train/Urban/images_png/2036.png  
 extracting: Train/Urban/images_png/2037.png  
 extracting: Train/Urban/images_png/2038.png  
 extracting: Train/Urban/images_png/2039.png  
 extracting: Train/Urban/images_png/2040.png  
 extracting: Train/Urban/images_png/2041.png  
 extracting: Train/Urban/images_png/2042.png  
 extracting: Train/Urban/images_png/2043.png  
 extracting: Train/Urban/images_png/2044.png  
 extracti

In [ ]:
!pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 74.4 MB/s eta 0:00:00


In [ ]:

import os
import shutil
from pathlib import Path
from tqdm import tqdm
import uuid
from ultralytics.data.converter import convert_segment_masks_to_yolo_seg

# Define paths
base_path = Path('/content')
train_rural_images = base_path / 'Train' / 'Rural' / 'images_png'
train_rural_masks = base_path / 'Train' / 'Rural' / 'masks_png'
train_urban_images = base_path / 'Train' / 'Urban' / 'images_png'
train_urban_masks = base_path / 'Train' / 'Urban' / 'masks_png'

val_rural_images = base_path / 'Val' / 'Rural' / 'images_png'
val_rural_masks = base_path / 'Val' / 'Rural' / 'masks_png'
val_urban_images = base_path / 'Val' / 'Urban' / 'images_png'
val_urban_masks = base_path / 'Val' / 'Urban' / 'masks_png'

# Create temporary directories for merged data
temp_train_images = base_path / 'temp_train_images'
temp_train_masks = base_path / 'temp_train_masks'
temp_val_images = base_path / 'temp_val_images'
temp_val_masks = base_path / 'temp_val_masks'

temp_train_images.mkdir(exist_ok=True)
temp_train_masks.mkdir(exist_ok=True)
temp_val_images.mkdir(exist_ok=True)
temp_val_masks.mkdir(exist_ok=True)

print(" Directories created successfully!")

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
✅ Directories created successfully!


In [ ]:
def copy_image_mask_pairs(images_dir, masks_dir, dest_images_dir, dest_masks_dir, prefix=""):
    """Copy image-mask pairs with the SAME UUID"""

    if not images_dir.exists() or not masks_dir.exists():
        print(f"⚠️ Warning: {images_dir} or {masks_dir} does not exist!")
        return 0

    image_files = list(images_dir.glob('*.png'))
    copied_count = 0

    for img_file in tqdm(image_files, desc=f"Copying {prefix}", leave=False):
        # Check if corresponding mask exists
        mask_file = masks_dir / img_file.name

        if mask_file.exists():
            # Generate ONE UUID for BOTH files
            unique_id = str(uuid.uuid4())[:8]
            new_filename = f"{prefix}_{unique_id}_{img_file.name}"

            # Copy image
            shutil.copy2(img_file, dest_images_dir / new_filename)

            # Copy mask with SAME UUID
            shutil.copy2(mask_file, dest_masks_dir / new_filename)

            copied_count += 1
        else:
            print(f"⚠️ Warning: No mask found for {img_file.name}")

    return copied_count

# Merge Train data
print("📁 Merging TRAIN data...")

print("🌾 Processing Train/Rural...")
rural_count = copy_image_mask_pairs(
    train_rural_images, train_rural_masks,
    temp_train_images, temp_train_masks,
    prefix="train_rural"
)
print(f"✓ Copied {rural_count} rural image-mask pairs")

print("🏙️ Processing Train/Urban...")
urban_count = copy_image_mask_pairs(
    train_urban_images, train_urban_masks,
    temp_train_images, temp_train_masks,
    prefix="train_urban"
)
print(f"✓ Copied {urban_count} urban image-mask pairs")

print(f"📊 Total Train: {rural_count + urban_count} image-mask pairs")

# Merge Val data
print("\n📁 Merging VAL data...")

print("🌾 Processing Val/Rural...")
val_rural_count = copy_image_mask_pairs(
    val_rural_images, val_rural_masks,
    temp_val_images, temp_val_masks,
    prefix="val_rural"
)
print(f"✓ Copied {val_rural_count} rural image-mask pairs")

print("🏙️ Processing Val/Urban...")
val_urban_count = copy_image_mask_pairs(
    val_urban_images, val_urban_masks,
    temp_val_images, temp_val_masks,
    prefix="val_urban"
)
print(f"✓ Copied {val_urban_count} urban image-mask pairs")

print(f"📊 Total Val: {val_rural_count + val_urban_count} image-mask pairs")

📁 Merging TRAIN data...
🌾 Processing Train/Rural...


✓ Copied 1366 rural image-mask pairs
🏙️ Processing Train/Urban...


✓ Copied 1156 urban image-mask pairs
📊 Total Train: 2522 image-mask pairs

📁 Merging VAL data...
🌾 Processing Val/Rural...


✓ Copied 992 rural image-mask pairs
🏙️ Processing Val/Urban...


✓ Copied 677 urban image-mask pairs
📊 Total Val: 1669 image-mask pairs


In [ ]:
# Create final YOLO dataset structure
yolo_dataset_path = base_path / 'LoveDA_YOLO'
yolo_images_train = yolo_dataset_path / 'images' / 'train'
yolo_images_val = yolo_dataset_path / 'images' / 'val'
yolo_labels_train = yolo_dataset_path / 'labels' / 'train'
yolo_labels_val = yolo_dataset_path / 'labels' / 'val'

yolo_images_train.mkdir(parents=True, exist_ok=True)
yolo_images_val.mkdir(parents=True, exist_ok=True)
yolo_labels_train.mkdir(parents=True, exist_ok=True)
yolo_labels_val.mkdir(parents=True, exist_ok=True)

print("✅ YOLO dataset structure created!")

✅ YOLO dataset structure created!


In [ ]:
print("🔄 Converting masks to YOLO format...")

print("📝 Converting TRAIN masks...")
convert_segment_masks_to_yolo_seg(
    masks_dir=str(temp_train_masks),
    output_dir=str(yolo_labels_train),
    classes=7
)
print("✓ Train masks converted!")

print("📝 Converting VAL masks...")
convert_segment_masks_to_yolo_seg(
    masks_dir=str(temp_val_masks),
    output_dir=str(yolo_labels_val),
    classes=7
)
print("✓ Val masks converted!")

Streaming output truncated to the last 5000 lines.
Processed and stored at /content/LoveDA_YOLO/labels/train/train_urban_ac7e34a4_1987.txt imgsz = 1024 x 1024
Processing /content/temp_train_masks/train_rural_d433b18d_675.png imgsz = 1024 x 1024
Processed and stored at /content/LoveDA_YOLO/labels/train/train_rural_d433b18d_675.txt imgsz = 1024 x 1024
Processing /content/temp_train_masks/train_urban_f1755e1d_1527.png imgsz = 1024 x 1024
Processed and stored at /content/LoveDA_YOLO/labels/train/train_urban_f1755e1d_1527.txt imgsz = 1024 x 1024
Processing /content/temp_train_masks/train_urban_f1e24a0f_2359.png imgsz = 1024 x 1024
Processed and stored at /content/LoveDA_YOLO/labels/train/train_urban_f1e24a0f_2359.txt imgsz = 1024 x 1024
Processing /content/temp_train_masks/train_urban_b88a799b_1396.png imgsz = 1024 x 1024
Processed and stored at /content/LoveDA_YOLO/labels/train/train_urban_b88a799b_1396.txt imgsz = 1024 x 1024
Processing /content/temp_train_masks/train_urban_5eb0cb62_2050.

In [ ]:
print("📋 Copying images to YOLO structure...")

print("🖼️ Copying TRAIN images...")
for img_file in tqdm(list(temp_train_images.glob('*.png')), desc="Train images"):
    shutil.copy2(img_file, yolo_images_train / img_file.name)

print("🖼️ Copying VAL images...")
for img_file in tqdm(list(temp_val_images.glob('*.png')), desc="Val images"):
    shutil.copy2(img_file, yolo_images_val / img_file.name)

print("✅ Images copied successfully!")

📋 Copying images to YOLO structure...
🖼️ Copying TRAIN images...


Train images: 100%|██████████| 2522/2522 [00:32<00:00, 76.64it/s]


🖼️ Copying VAL images...


Val images: 100%|██████████| 1669/1669 [00:20<00:00, 81.75it/s]

✅ Images copied successfully!


In [ ]:
yaml_content = """# LoveDA Dataset Configuration for YOLO Segmentation
path: /content/LoveDA_YOLO
train: images/train
val: images/val

names:
  0: background
  1: building
  2: road
  3: water
  4: barren
  5: forest
  6: agriculture

nc: 7
"""

yaml_path = yolo_dataset_path / 'data.yaml'
with open(yaml_path, 'w') as f:
    f.write(yaml_content)

print(f"✅ data.yaml created at: {yaml_path}")

✅ data.yaml created at: /content/LoveDA_YOLO/data.yaml


In [ ]:
print("🧹 Cleaning up temporary files...")
shutil.rmtree(temp_train_images)
shutil.rmtree(temp_train_masks)
shutil.rmtree(temp_val_images)
shutil.rmtree(temp_val_masks)
print("✅ Temporary files removed!")

🧹 Cleaning up temporary files...
✅ Temporary files removed!


In [ ]:
train_images = set([f.stem for f in yolo_images_train.glob('*.png')])
train_labels = set([f.stem for f in yolo_labels_train.glob('*.txt')])
val_images = set([f.stem for f in yolo_images_val.glob('*.png')])
val_labels = set([f.stem for f in yolo_labels_val.glob('*.txt')])

print("=" * 60)
print("📊 DATASET VERIFICATION")
print("=" * 60)
print(f"\nTRAIN SET:")
print(f"  Images: {len(train_images)}")
print(f"  Labels: {len(train_labels)}")
print(f"  Matched: {len(train_images & train_labels)}/{len(train_images)}")

print(f"\nVAL SET:")
print(f"  Images: {len(val_images)}")
print(f"  Labels: {len(val_labels)}")
print(f"  Matched: {len(val_images & val_labels)}/{len(val_images)}")

if len(train_images & train_labels) == len(train_images) and len(val_images & val_labels) == len(val_images):
    print("\n✅ Perfect match! Dataset ready for training!")
else:
    print("\n⚠️ Mismatch detected!")

📊 DATASET VERIFICATION

TRAIN SET:
  Images: 2522
  Labels: 2522
  Matched: 2522/2522

VAL SET:
  Images: 1669
  Labels: 1669
  Matched: 1669/1669

✅ Perfect match! Dataset ready for training!


In [ ]:
import random
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from PIL import Image
from pathlib import Path

# Set seed for reproducibility
random.seed(42)
np.random.seed(42)

# Define class names and colors
CLASS_INFO = {
    0: {'name': 'Background', 'color': [0, 0, 0]},           # Black
    1: {'name': 'Building', 'color': [255, 0, 0]},           # Red
    2: {'name': 'Road', 'color': [255, 255, 0]},             # Yellow
    3: {'name': 'Water', 'color': [0, 0, 255]},              # Blue
    4: {'name': 'Barren', 'color': [128, 64, 0]},            # Brown
    5: {'name': 'Forest', 'color': [0, 255, 0]},             # Green
    6: {'name': 'Agriculture', 'color': [255, 165, 0]},      # Orange
}

def mask_to_color(mask, class_info):
    """Convert grayscale mask to RGB colored mask"""
    h, w = mask.shape
    color_mask = np.zeros((h, w, 3), dtype=np.uint8)

    for class_id, info in class_info.items():
        color_mask[mask == class_id] = info['color']

    return color_mask

def visualize_samples(split='train', num_samples=10, figsize=(20, 40)):
    """
    Visualize random samples from the dataset

    Args:
        split: 'train' or 'val'
        num_samples: number of samples to visualize
        figsize: figure size for matplotlib
    """
    # Get paths
    base_path = Path('/content')

    # Define source paths based on split
    if split == 'train':
        rural_images = base_path / 'Train' / 'Rural' / 'images_png'
        rural_masks = base_path / 'Train' / 'Rural' / 'masks_png'
        urban_images = base_path / 'Train' / 'Urban' / 'images_png'
        urban_masks = base_path / 'Train' / 'Urban' / 'masks_png'
    else:  # val
        rural_images = base_path / 'Val' / 'Rural' / 'images_png'
        rural_masks = base_path / 'Val' / 'Rural' / 'masks_png'
        urban_images = base_path / 'Val' / 'Urban' / 'images_png'
        urban_masks = base_path / 'Val' / 'Urban' / 'masks_png'

    # Collect all image paths with their corresponding masks and domain
    all_samples = []

    # Rural samples
    if rural_images.exists():
        for img_path in rural_images.glob('*.png'):
            mask_path = rural_masks / img_path.name
            if mask_path.exists():
                all_samples.append({
                    'image': img_path,
                    'mask': mask_path,
                    'domain': 'Rural'
                })

    # Urban samples
    if urban_images.exists():
        for img_path in urban_images.glob('*.png'):
            mask_path = urban_masks / img_path.name
            if mask_path.exists():
                all_samples.append({
                    'image': img_path,
                    'mask': mask_path,
                    'domain': 'Urban'
                })

    print(f"Found {len(all_samples)} samples in {split} set")

    # Sample random images
    if len(all_samples) < num_samples:
        print(f"⚠️ Only {len(all_samples)} samples available, showing all")
        num_samples = len(all_samples)

    selected_samples = random.sample(all_samples, num_samples)

    # Create figure
    fig, axes = plt.subplots(num_samples, 2, figsize=figsize)

    if num_samples == 1:
        axes = axes.reshape(1, -1)

    # Plot each sample
    for idx, sample in enumerate(selected_samples):
        # Load image and mask
        image = np.array(Image.open(sample['image']))
        mask = np.array(Image.open(sample['mask']))

        # Convert mask to colored version
        colored_mask = mask_to_color(mask, CLASS_INFO)

        # Get unique classes in this mask
        unique_classes = np.unique(mask)
        classes_present = [CLASS_INFO[c]['name'] for c in unique_classes if c in CLASS_INFO]

        # Plot original image
        axes[idx, 0].imshow(image)
        axes[idx, 0].set_title(
            f"{sample['domain']} - {sample['image'].name}\n"
            f"Classes: {', '.join(classes_present)}",
            fontsize=10
        )
        axes[idx, 0].axis('off')

        # Plot colored mask
        axes[idx, 1].imshow(colored_mask)
        axes[idx, 1].set_title(f"Segmentation Mask", fontsize=10)
        axes[idx, 1].axis('off')

    plt.tight_layout(rect=[0, 0, 0.85, 1])  # Leave space for legend

    # Create legend
    legend_patches = []
    for class_id, info in CLASS_INFO.items():
        color_normalized = [c/255.0 for c in info['color']]
        patch = mpatches.Patch(color=color_normalized, label=f"{class_id}: {info['name']}")
        legend_patches.append(patch)

    # Add legend to the right side of the figure
    fig.legend(
        handles=legend_patches,
        loc='center right',
        fontsize=12,
        title='Class Legend',
        title_fontsize=14,
        frameon=True,
        fancybox=True,
        shadow=True
    )

    plt.suptitle(
        f'LoveDA Dataset - {split.upper()} Split (Random Samples with seed=42)',
        fontsize=16,
        fontweight='bold',
        y=0.995
    )

    plt.show()

    # Print statistics
    print("\n" + "="*60)
    print("STATISTICS FOR SELECTED SAMPLES")
    print("="*60)

    domain_count = {'Rural': 0, 'Urban': 0}
    class_frequency = {i: 0 for i in range(7)}

    for sample in selected_samples:
        domain_count[sample['domain']] += 1
        mask = np.array(Image.open(sample['mask']))
        unique_classes = np.unique(mask)
        for c in unique_classes:
            if c in class_frequency:
                class_frequency[c] += 1

    print(f"\n📊 Domain Distribution:")
    print(f"   Rural: {domain_count['Rural']}/{num_samples} ({domain_count['Rural']/num_samples*100:.1f}%)")
    print(f"   Urban: {domain_count['Urban']}/{num_samples} ({domain_count['Urban']/num_samples*100:.1f}%)")

    print(f"\n🎨 Class Frequency (images containing each class):")
    for class_id in sorted(class_frequency.keys()):
        count = class_frequency[class_id]
        percentage = (count / num_samples) * 100
        print(f"   {CLASS_INFO[class_id]['name']:12} : {count}/{num_samples} ({percentage:.1f}%)")

# Visualize 10 random train samples
print("🖼️ VISUALIZING TRAIN SAMPLES")
print("="*60)
visualize_samples(split='train', num_samples=10, figsize=(20, 40))

# Visualize 10 random val samples
print("\n\n🖼️ VISUALIZING VAL SAMPLES")
print("="*60)
visualize_samples(split='val', num_samples=10, figsize=(20, 40))

Output hidden; open in https://colab.research.google.com to view.

In [ ]:
# !rm -rf /content/Test /content/Train

In [ ]:
# !unzip /content/Train.zip
# !unzip /content/Val.zip

In [ ]:
# !rm -rf /content/Train /content/Val

In [ ]:
from ultralytics import YOLO
model = YOLO("yolo26n-seg.pt")
model

YOLO(
  (model): SegmentationModel(
    (model): Sequential(
      (0): Conv(
        (conv): Conv2d(3, 16, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
        (bn): BatchNorm2d(16, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
        (act): SiLU(inplace=True)
      )
      (1): Conv(
        (conv): Conv2d(16, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
        (bn): BatchNorm2d(32, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
        (act): SiLU(inplace=True)
      )
      (2): C3k2(
        (cv1): Conv(
          (conv): Conv2d(32, 32, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn): BatchNorm2d(32, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
          (act): SiLU(inplace=True)
        )
        (cv2): Conv(
          (conv): Conv2d(48, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn): BatchNorm2d(64, eps=0.001, momentum=0.03, affine=True, track_runni

In [ ]:
results = model.train(data=yaml_path, epochs=30, imgsz=640, batch = 20)

Ultralytics 8.4.14 🚀 Python-3.12.12 torch-2.9.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=20, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/LoveDA_YOLO/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=30, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26n-seg.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, perspec

In [ ]:
# model.export(format="onnx")

In [ ]:
from ultralytics import YOLO
model = YOLO("/content/runs/segment/train/weights/best.pt")

In [ ]:
metrics = model.val()

Ultralytics 8.4.14 🚀 Python-3.12.12 torch-2.9.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLO26n-seg summary (fused): 139 layers, 2,690,249 parameters, 0 gradients, 9.0 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1526.9±1360.0 MB/s, size: 1428.9 KB)
val: Scanning /content/LoveDA_YOLO/labels/val.cache... 1669 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1669/1669 500.0Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 105/105 1.0it/s 1:42
                   all       1669      49386      0.409      0.308      0.278      0.153      0.372      0.275      0.241      0.108
            background       1645      12348      0.425      0.146      0.147     0.0816       0.19     0.0607     0.0391     0.0114
              building       1087      14361      0.474      0.534      0.492      0.239      0.475      0.498      0.461      0.189
                  road       1259 

In [ ]:
print("📊 Overall Metrics")
print("mAP@0.5      :", metrics.box.map50)
print("mAP@0.5:0.95 :", metrics.box.map)
print("Precision    :", metrics.box.mp)
print("Recall       :", metrics.box.mr)

print("\n📊 Per-class mAP@0.5")
print(metrics.box.maps)  # array per class

📊 Overall Metrics
mAP@0.5      : 0.27830840732470896
mAP@0.5:0.95 : 0.1533958492740898
Precision    : 0.4092845499496965
Recall       : 0.30830667227911823

📊 Per-class mAP@0.5
[   0.081566     0.23904     0.16754     0.29324     0.08248    0.089875     0.12003]


In [ ]:
!zip -r satellite_model_run.zip /content/runs/

  adding: content/runs/ (stored 0%)
  adding: content/runs/segment/ (stored 0%)
  adding: content/runs/segment/val/ (stored 0%)
  adding: content/runs/segment/val/MaskR_curve.png (deflated 10%)
  adding: content/runs/segment/val/val_batch0_labels.jpg (deflated 4%)
  adding: content/runs/segment/val/val_batch1_pred.jpg (deflated 4%)
  adding: content/runs/segment/val/val_batch1_labels.jpg (deflated 4%)
  adding: content/runs/segment/val/confusion_matrix_normalized.png (deflated 19%)
  adding: content/runs/segment/val/MaskF1_curve.png (deflated 10%)
  adding: content/runs/segment/val/val_batch2_labels.jpg (deflated 5%)
  adding: content/runs/segment/val/BoxPR_curve.png (deflated 7%)
  adding: content/runs/segment/val/val_batch0_pred.jpg (deflated 4%)
  adding: content/runs/segment/val/BoxF1_curve.png (deflated 10%)
  adding: content/runs/segment/val/confusion_matrix.png (deflated 16%)
  adding: content/runs/segment/val/BoxP_curve.png (deflated 6%)
  adding: content/runs/segment/val/BoxR_

In [ ]:
!mkdir -p /content/drive/MyDrive/satellite/
!rm -rf /content/drive/MyDrive/satellite/satellite_model_run.zip
!cp /content/satellite_model_run.zip /content/drive/MyDrive/satellite/